# Assign Unmatched RL Roles to SASB Categories

**Two-step approach (at `role_k10000_v3` level):**
1. **Step 1 (LLM role-based):** `gemma_role_classification_output.csv` — gemma classified each unique role triple by title. Group by `role_k10000_v3`, union all assigned SASB categories.
2. **Step 2 (description-based fallback):** For `role_k10000_v3` labels that step 1 never mapped, use `gemma_combined_classification_output_100k.csv` (10 jobs per `role_k10000_v3`). Assign a SASB category if hit rate ≥ `MIN_RATE` (e.g. ≥3 out of 10 descriptions tagged).

**Output:** `role_to_sasb_mapping.csv` — one row per `role_k10000_v3`.

In [3]:
import ast
import pandas as pd
from pathlib import Path

BASE      = Path('D:/Dropbox/fengheliu/temp/sasb_jobs')
TEMP_DATA = BASE / 'temp_data'

ROLE_LLM_OUT = TEMP_DATA / 'step1_gemma_role_classification' / 'gemma_role_classification_output.csv'
DESC_LLM_OUT = TEMP_DATA / 'step2_gemma_combined_classification' / 'gemma_combined_classification_output_100k.csv'
SAMPLE       = TEMP_DATA / 'step2_gemma_combined_classification' / 'role_stratified_sample.csv'
OUTPUT       = TEMP_DATA / 'step3_assign_unmatched_roles' / 'role_to_sasb_mapping.csv'

ROLE_COL = 'role_k10000_v3'

# With 10 jobs per role, MIN_RATE=0.20 means >=2/10 descriptions tagged
# MIN_RATE=0.30 means >=3/10 — more conservative
MIN_N    = 10     # minimum jobs with that role in the 100k sample (handles edge cases)
MIN_RATE = 0.30  # fraction of descriptions that must be tagged

def parse_list(val):
    if isinstance(val, list): return val
    if isinstance(val, str) and val.strip():
        try: return ast.literal_eval(val)
        except: return []
    return []

## EDA — gemma_combined_classification_output_100k.csv

In [4]:
eda = pd.read_csv(DESC_LLM_OUT)
eda['sasb_categories'] = eda['sasb_categories'].apply(parse_list)
eda['n_cats']          = eda['sasb_categories'].apply(len)

total = len(eda)
n_pos = eda['n_cats'].gt(0).sum()
print(f"Total rows            : {total:,}")
print(f"Overall hit rate      : {n_pos:,} / {total:,} = {n_pos/total*100:.2f}%")
print(f"Avg categories (pos)  : {eda.loc[eda['n_cats']>0,'n_cats'].mean():.2f}")
print()

# Distribution of n_cats per job
print("Distribution of #categories assigned per job:")
print(eda['n_cats'].value_counts().sort_index().rename('n_jobs').to_frame().to_string())
print()

# Hit rate per SASB category
from collections import Counter
cat_counts = Counter(cat for cats in eda['sasb_categories'] for cat in cats)
cat_df = (
    pd.DataFrame.from_dict(cat_counts, orient='index', columns=['n_jobs'])
    .sort_values('n_jobs', ascending=False)
    .assign(hit_rate_pct=lambda d: (d['n_jobs'] / total * 100).round(3))
)
cat_df.index.name = 'sasb_category'
print(f"Hit rate by SASB category (out of {total:,} total jobs):")
display(cat_df)
print()

# Distribution of number of jobs per role (should be ~10 each)
sample_eda = pd.read_csv(SAMPLE, usecols=['position_id', ROLE_COL])
jobs_per_role = (
    sample_eda.merge(eda[['position_id']], on='position_id', how='inner')
    [ROLE_COL].value_counts()
)
print(f"Jobs per {ROLE_COL} — summary (expected ~10 each):")
print(jobs_per_role.describe().to_string())
print()
print("Distribution of jobs-per-role counts:")
print(jobs_per_role.value_counts().sort_index().rename('n_roles').to_frame().to_string())


Total rows            : 99,843
Overall hit rate      : 23,792 / 99,843 = 23.83%
Avg categories (pos)  : 1.12

Distribution of #categories assigned per job:
        n_jobs
n_cats        
0        76051
1        21062
2         2526
3          178
4           23
5            3

Hit rate by SASB category (out of 99,843 total jobs):


,n_jobs,hit_rate_pct
sasb_category,,
Employee_Health_&_Safety,4371,4.378
Business_Ethics,2541,2.545
Data_Security,2209,2.212
Product_Quality_&_Safety,2010,2.013
Supply_Chain_Management,1981,1.984
Human_Rights_&_Community_Relations,1540,1.542
Energy_Management,1469,1.471
Waste_&_Hazardous_Materials_Management,1272,1.274
"Employee_Engagement,_Diversity_&_Inclusion",1198,1.200



Jobs per role_k10000_v3 — summary (expected ~10 each):
count    10000.000000
mean         9.984300
std          0.314743
min          1.000000
25%         10.000000
50%         10.000000
75%         10.000000
max         10.000000

Distribution of jobs-per-role counts:
       n_roles
count         
1            6
2            1
3            2
4            3
5            5
6            4
7            3
8            4
9            5
10        9967


## Step 1 — Role-title LLM mapping
Group `gemma_role_classification_output.csv` by `role_k10000_v3`, union all assigned SASB categories.

In [5]:
role_df = pd.read_csv(ROLE_LLM_OUT)
role_df['sasb_categories'] = role_df['sasb_categories'].apply(parse_list)
role_df = role_df[role_df[ROLE_COL].notna()]

print(f'Loaded {len(role_df):,} rows, {role_df[ROLE_COL].nunique():,} unique {ROLE_COL}')

step1 = (
    role_df.groupby(ROLE_COL)['sasb_categories']
    .agg(lambda x: sorted({cat for cats in x for cat in cats}))
    .reset_index()
)
step1['source'] = step1['sasb_categories'].apply(lambda x: 'gemma_role' if x else 'unmatched')

matched   = step1[step1['source'] == 'gemma_role']
unmatched = step1[step1['source'] == 'unmatched']

print(f'Step 1 — matched   : {len(matched):,} role labels')
print(f'Step 1 — unmatched : {len(unmatched):,} role labels')
matched.head(10)

Loaded 10,002 rows, 10,002 unique role_k10000_v3
Step 1 — matched   : 2,411 role labels
Step 1 — unmatched : 7,591 role labels


,role_k10000_v3,sasb_categories,source
3,"""IoT Product Manager""",[Product_Design_&_Lifecycle_Management],gemma_role
7,3D Artist,[Product_Design_&_Lifecycle_Management],gemma_role
8,3D Designer,[Product_Design_&_Lifecycle_Management],gemma_role
9,3D Printing Engineer,[Materials_Sourcing_&_Efficiency],gemma_role
30,AML Analyst,[Business_Ethics],gemma_role
53,Academic Coach,[Employee_Health_&_Safety],gemma_role
63,Access Administrator,[Data_Security],gemma_role
64,Access Analyst,[Access_&_Affordability],gemma_role
65,Access Control Analyst,[Data_Security],gemma_role
67,Access Control Technician,[Data_Security],gemma_role


## Step 2 — Description-based fallback (100k sample)
Join `role_stratified_sample.csv` with `gemma_combined_classification_output_100k.csv` on `position_id`.
For each unmatched `role_k10000_v3`, compute per-SASB hit rate and assign if ≥ `MIN_RATE`.

In [6]:
sample   = pd.read_csv(SAMPLE, usecols=['position_id', ROLE_COL])
desc_llm = pd.read_csv(DESC_LLM_OUT, usecols=['position_id', 'sasb_categories'])
desc_llm['sasb_categories'] = desc_llm['sasb_categories'].apply(parse_list)

merged = sample.merge(desc_llm, on='position_id', how='inner')
print(f'Joined: {len(merged):,} rows, {merged[ROLE_COL].nunique():,} unique {ROLE_COL}')

unmatched_roles = set(unmatched[ROLE_COL])
df_unmatched    = merged[merged[ROLE_COL].isin(unmatched_roles)].copy()
print(f'Unmatched roles in sample : {df_unmatched[ROLE_COL].nunique():,} labels, {len(df_unmatched):,} jobs')

all_cats = sorted({cat for cats in df_unmatched['sasb_categories'] for cat in cats})
print(f'SASB categories seen      : {len(all_cats)}')

Joined: 99,843 rows, 10,000 unique role_k10000_v3
Unmatched roles in sample : 7,589 labels, 75,783 jobs
SASB categories seen      : 26


In [7]:
records = []
for role, grp in df_unmatched.groupby(ROLE_COL, dropna=False):
    n = len(grp)
    cat_rates = {
        cat: grp['sasb_categories'].apply(lambda x: cat in x).sum() / n
        for cat in all_cats
    }
    top_cat  = max(cat_rates, key=cat_rates.get) if cat_rates else None
    top_rate = cat_rates[top_cat] if top_cat else 0.0

    if n >= MIN_N:
        assigned = sorted(cat for cat, rate in cat_rates.items() if rate >= MIN_RATE)
        source   = 'gemma_desc' if assigned else 'below_threshold'
    else:
        assigned = []
        source   = 'too_few_samples'

    records.append({
        ROLE_COL:          role,
        'n_jobs':          n,
        'top_cat':         top_cat,
        'top_rate':        round(top_rate, 4),
        'sasb_categories': assigned,
        'source':          source,
    })

step2 = pd.DataFrame(records)
print(f'Step 2 — desc-assigned   : {(step2["source"] == "gemma_desc").sum()}')
print(f'Step 2 — too few samples : {(step2["source"] == "too_few_samples").sum()}')
print(f'Step 2 — below threshold : {(step2["source"] == "below_threshold").sum()}')
step2[step2['source'] == 'gemma_desc'].sort_values('top_rate', ascending=False).head(20)

Step 2 — desc-assigned   : 784
Step 2 — too few samples : 20
Step 2 — below threshold : 6785


,role_k10000_v3,n_jobs,top_cat,top_rate,sasb_categories,source
7432,Waste Coordinator,10,Waste_&_Hazardous_Materials_Management,1.0,[Waste_&_Hazardous_Materials_Management],gemma_desc
7343,Violence Prevention Advocate,10,Human_Rights_&_Community_Relations,1.0,[Human_Rights_&_Community_Relations],gemma_desc
3674,International Supply Chain Analyst,10,Supply_Chain_Management,1.0,[Supply_Chain_Management],gemma_desc
4416,Medical Supply Operator,10,Supply_Chain_Management,1.0,[Supply_Chain_Management],gemma_desc
2882,Fraud Prevention,10,Business_Ethics,1.0,[Business_Ethics],gemma_desc
5555,Psychiatric Technician,10,Employee_Health_&_Safety,1.0,[Employee_Health_&_Safety],gemma_desc
2084,Diversity Coordinator,10,"Employee_Engagement,_Diversity_&_Inclusion",1.0,"[Employee_Engagement,_Diversity_&_Inclusion]",gemma_desc
1815,Data Security Analyst,10,Data_Security,1.0,"[Customer_Privacy, Data_Security]",gemma_desc
1804,Data Protection Engineer,10,Data_Security,1.0,[Data_Security],gemma_desc
2343,Energy Advisor,10,Energy_Management,1.0,[Energy_Management],gemma_desc


In [17]:
step2[step2['source'] == 'gemma_desc'].sort_values('top_rate', ascending=False).tail(25)

,role_k10000_v3,n_jobs,top_cat,top_rate,sasb_categories,source
7314,Veterinary Project Coordinator,10,Employee_Health_&_Safety,0.3,[Employee_Health_&_Safety],gemma_desc
391,Asset Protection,10,Business_Ethics,0.3,"[Business_Ethics, Employee_Health_&_Safety]",gemma_desc
388,Asset Integrity Engineer,10,Critical_Incident_Risk_Management,0.3,"[Critical_Incident_Risk_Management, Physical_I...",gemma_desc
377,Assembly Operations Manager,10,Employee_Health_&_Safety,0.3,[Employee_Health_&_Safety],gemma_desc
373,Assembly Engineer,10,Product_Design_&_Lifecycle_Management,0.3,[Product_Design_&_Lifecycle_Management],gemma_desc
344,Area Operations Management,10,Labor_Practices,0.3,[Labor_Practices],gemma_desc
332,Archaeology Technician,10,Ecological_Impacts,0.3,[Ecological_Impacts],gemma_desc
302,Apparel Product Manager,10,Product_Design_&_Lifecycle_Management,0.3,[Product_Design_&_Lifecycle_Management],gemma_desc
300,Apparel Management,10,Supply_Chain_Management,0.3,[Supply_Chain_Management],gemma_desc
7568,Youth Camp Coordinator,10,Employee_Health_&_Safety,0.3,[Employee_Health_&_Safety],gemma_desc


## Inspect: roles near but below the threshold

In [20]:
near = step2[
    (step2['source'] == 'below_threshold') &
    (step2['top_rate'] >= 0.10)
].sort_values('top_rate', ascending=False)

print(f'Roles below threshold but top_rate >= 10%: {len(near)}')
near[['role_k10000_v3', 'n_jobs', 'top_cat', 'top_rate']].head(15)

Roles below threshold but top_rate >= 10%: 4774


,role_k10000_v3,n_jobs,top_cat,top_rate
7564,Yoga Teacher,10,Employee_Health_&_Safety,0.2
7519,Women's Health Nurse,10,Employee_Health_&_Safety,0.2
45,Academic Advising Director,10,Access_&_Affordability,0.2
46,Academic Assistant,10,Management_of_the_Legal_&_Regulatory_Environment,0.2
57,Access Control Officer,10,Employee_Health_&_Safety,0.2
20,AI Researcher,10,Data_Security,0.2
7520,Women's Health Sales,10,Customer_Welfare,0.2
7525,Woodworking Supervisor,10,Employee_Health_&_Safety,0.2
7528,Work Permit Analyst,10,Employee_Health_&_Safety,0.2
7530,Workers' Compensation Attorney,10,Employee_Health_&_Safety,0.2


## Threshold sensitivity (how many roles get mapped at each cutoff)

In [10]:
eligible = step2[step2['n_jobs'] >= MIN_N]

rows = []
for thresh in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
    n_desc = (eligible['top_rate'] >= thresh).sum()
    rows.append({
        'min_rate':     thresh,
        'desc_assigned': n_desc,
        'total_mapped': len(matched) + n_desc,
        'pct_of_unmatched': f"{n_desc/len(eligible)*100:.1f}%" if len(eligible) else 'N/A'
    })

pd.DataFrame(rows)

,min_rate,desc_assigned,total_mapped,pct_of_unmatched
0,0.1,5566,7977,73.4%
1,0.2,1929,4340,25.4%
2,0.3,786,3197,10.4%
3,0.4,351,2762,4.6%
4,0.5,173,2584,2.3%
5,0.6,93,2504,1.2%
6,0.7,66,2477,0.9%
7,0.8,38,2449,0.5%


## Combine Step 1 + Step 2 → final mapping

In [11]:
role_counts = merged[ROLE_COL].value_counts().rename('n_jobs_100k')

step1_out = matched[[ROLE_COL, 'sasb_categories', 'source']].copy()
step1_out['n_jobs'] = step1_out[ROLE_COL].map(role_counts)

step2_out = step2[[ROLE_COL, 'n_jobs', 'sasb_categories', 'source']].copy()

mapping = pd.concat(
    [step1_out[[ROLE_COL, 'n_jobs', 'sasb_categories', 'source']],
     step2_out],
    ignore_index=True
).sort_values(ROLE_COL).reset_index(drop=True)

has_mapping = mapping['sasb_categories'].apply(len) > 0
print(f'Total {ROLE_COL} labels   : {len(mapping):,}')
print(f'  With SASB mapping     : {has_mapping.sum():,} ({has_mapping.mean()*100:.1f}%)')
print(f'    from gemma_role     : {(mapping["source"]=="gemma_role").sum():,}')
print(f'    from gemma_desc     : {(mapping["source"]=="gemma_desc").sum():,}')
print(f'  No mapping            : {(~has_mapping).sum():,} ({(~has_mapping).mean()*100:.1f}%)')

mapping[has_mapping].head(20)

Total role_k10000_v3 labels   : 10,000
  With SASB mapping     : 3,197 (32.0%)
    from gemma_role     : 2,411
    from gemma_desc     : 786
  No mapping            : 6,803 (68.0%)


,role_k10000_v3,n_jobs,sasb_categories,source
2,"""Investment Banking HR""",10,"[Employee_Engagement,_Diversity_&_Inclusion]",gemma_desc
3,"""IoT Product Manager""",10,[Product_Design_&_Lifecycle_Management],gemma_role
7,3D Artist,10,[Product_Design_&_Lifecycle_Management],gemma_role
8,3D Designer,10,[Product_Design_&_Lifecycle_Management],gemma_role
9,3D Printing Engineer,10,[Materials_Sourcing_&_Efficiency],gemma_role
22,AI Project Manager,10,[Data_Security],gemma_desc
30,AML Analyst,10,[Business_Ethics],gemma_role
34,API Developer,10,[Data_Security],gemma_desc
36,API Integration Specialist,10,[Data_Security],gemma_desc
53,Academic Coach,10,[Employee_Health_&_Safety],gemma_role


## Save

In [12]:
mapping.to_csv(OUTPUT, index=False)
print(f'Saved {len(mapping):,} rows to: {OUTPUT}')
mapping[has_mapping]

Saved 10,000 rows to: D:\Dropbox\fengheliu\temp\sasb_jobs\role_to_sasb_mapping.csv


,role_k10000_v3,n_jobs,sasb_categories,source
2,"""Investment Banking HR""",10,"[Employee_Engagement,_Diversity_&_Inclusion]",gemma_desc
3,"""IoT Product Manager""",10,[Product_Design_&_Lifecycle_Management],gemma_role
7,3D Artist,10,[Product_Design_&_Lifecycle_Management],gemma_role
8,3D Designer,10,[Product_Design_&_Lifecycle_Management],gemma_role
9,3D Printing Engineer,10,[Materials_Sourcing_&_Efficiency],gemma_role
...,...,...,...,...
9984,Youth Worker,10,[Human_Rights_&_Community_Relations],gemma_desc
9985,Zoning Analyst,10,[Ecological_Impacts],gemma_role
9986,Zoo Maintenance Technician,10,[Ecological_Impacts],gemma_role
9987,Zoo Supervisor,10,[Employee_Health_&_Safety],gemma_role
